# Load sample retail data into bronze

Loads `sample_data/*.csv` into the lakehouse as bronze Delta tables. Run this in a Fabric Notebook attached to `lh_retail_sales_gold`.

In [ ]:
import os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DecimalType

# Adjust to where your CSVs land in OneLake / ABFS
BASE = 'Files/sample_data'
BRONZE_DB = 'bronze'

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {BRONZE_DB}')

customer_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('customer_name', StringType(), False),
    StructField('customer_segment', StringType(), False),
    StructField('country', StringType(), False),
    StructField('region', StringType(), False),
    StructField('signup_date', DateType(), False),
])
(spark.read.option('header', True).schema(customer_schema).csv(f'{BASE}/customers.csv')
    .write.mode('overwrite').saveAsTable(f'{BRONZE_DB}.customers_raw'))

In [ ]:
product_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('product_name', StringType(), False),
    StructField('category', StringType(), False),
    StructField('subcategory', StringType(), False),
    StructField('list_price', DecimalType(18, 2), False),
    StructField('cost_price', DecimalType(18, 2), False),
])
(spark.read.option('header', True).schema(product_schema).csv(f'{BASE}/products.csv')
    .write.mode('overwrite').saveAsTable(f'{BRONZE_DB}.products_raw'))

In [ ]:
sales_schema = StructType([
    StructField('order_id', StringType(), False),
    StructField('customer_id', StringType(), False),
    StructField('product_id', StringType(), False),
    StructField('order_date', DateType(), False),
    StructField('ship_date', DateType(), False),
    StructField('quantity', IntegerType(), False),
    StructField('unit_price', DecimalType(18, 2), False),
    StructField('discount_pct', DecimalType(5, 4), False),
])
(spark.read.option('header', True).schema(sales_schema).csv(f'{BASE}/sales.csv')
    .write.mode('overwrite').saveAsTable(f'{BRONZE_DB}.sales_raw'))

In [ ]:
spark.sql(f'SELECT COUNT(*) AS rows FROM {BRONZE_DB}.customers_raw').show()
spark.sql(f'SELECT COUNT(*) AS rows FROM {BRONZE_DB}.products_raw').show()
spark.sql(f'SELECT COUNT(*) AS rows FROM {BRONZE_DB}.sales_raw').show()